# ClassifAI Webinar - Additional Features

The three modules of ClassifAI, Vectorisers, VectorStore and Servers, make up the core functionality of the Package, facilitating Semantic Search.

We're now going to cover some additional features that expand the use cases and possiblities using ClassifAI.

In [ ]:
from classifai.vectorisers import HuggingFaceVectoriser
from classifai.indexers import VectorStore
from classifai.indexers.dataclasses import VectorStoreSearchInput


#create a vectorise used for converting text to vectors
demo_vectoriser = HuggingFaceVectoriser(model_name="sentence-transformers/all-MiniLM-L6-v2")

#reload one of the vectorstores we created in the previous example, which can be 'searched' to find coded data that is similar to an uncoded sample
demo_vectorstore = VectorStore(
    file_name="./mock_coding_dataset.csv",
    data_type="csv",
    vectoriser=demo_vectoriser,
    output_dir="./saved_vectorstore/",
    overwrite=True
)

#create a search input with an id and a query, which is the text we want to find similar coded data for
search_input = VectorStoreSearchInput({'id':[1], 'query':["rock climbing instructor!!"]})

#call the vectorstore search method passing the input search onbject, returns a shortlist of similar candidate coded entries.
demo_vectorstore.search(search_input, n_results=5)

## Hooks

Hooks provide a way to modify the traditional flow of data through the VectorStore pipeline, by manipulating the content of the input and output dataclasses

![Hook Workflow](./images/pre_and_post_search_hooks.png)

<b></b>

In [ ]:
from classifai.indexers.hooks.default_hooks import CapitalisationStandardisingHook, DeduplicationHook

cap_hook = CapitalisationStandardisingHook(method="upper", colname="query")
dedup_hook = DeduplicationHook(score_aggregation_method="max")

In [ ]:
vector_store_with_hooks = VectorStore(
    file_name="mock_coding_dataset.csv",
    data_type="csv",
    vectoriser=demo_vectoriser,
    output_dir="./vectorstore_with_hooks",
    hooks={
        "search_preprocess": cap_hook,
        "search_postprocess": [dedup_hook],
    },
    overwrite=True
)



search_input = VectorStoreSearchInput({'id':[1], 'query':["rock climbing instructor!!"]})

vector_store_with_hooks.search(search_input, n_results=5)

<b></b>

![Hook Workflow](./images/search_spellcheck_hook.svg)

<b></b>

Users can also write their own hooks, either directly as a function or by <b>inheriting from the hook base class</b>

In [ ]:
from classifai.indexers.hooks import HookBase





#defining a simple function as a hook, accepting the dataframe as an argument and returing a modified dataframe
def removePunctuation(df):
    import string
    df['query'] = df['query'].str.translate(str.maketrans('', '', string.punctuation))
    return df







#defining a hook class, which inherits from the HookBase
class RemovePunctuationHook(HookBase):
    def __init__(self, colname):
        self.colname = colname

    def __call__(self, df):
        import string
        df[self.colname] = df[self.colname].str.translate(str.maketrans('', '', string.punctuation))
        return df
    
punct_hook = RemovePunctuationHook(colname="query")



In [ ]:
vector_store_with_even_more_hooks = VectorStore(
    file_name="./mock_coding_dataset.csv",
    data_type="csv",
    vectoriser=demo_vectoriser,
    output_dir="./vectorstore_with_hooks",
    hooks={
        "search_preprocess": [cap_hook, removePunctuation],
        "search_postprocess": [dedup_hook],
    },
    overwrite=True
)


#create a search input with an id and a query, which is the text we want to find similar coded data for
search_input = VectorStoreSearchInput({'id':[1], 'query':["rock climbing instructor!!!!!"]})

vector_store_with_even_more_hooks.search(search_input, n_results=5)

<b></b>

## AI Agents

ClassifAI provides a pre-made hook to use AI agents, allowing the user to design how the AI agent interacts with the output search dataclass of the Vectorstore. This is set up to allow the user to construct an AI agent that can manipulate the output data following user instructions.

![Rag agent in action](./images/rag_agent_at_work.png)

In [ ]:


# CLASSIFICATION INSTRUCTIONS FOR AI AGENT
CONTEXT_PROMPT_CLASSIFICATION = """You are an agent helping classify job descriptions.
Your task is to identify the most relevant retrieved document for a given query.
You will receive a Pandas DataFrame, with a schema described in an Input Format
section, which is converted to JSON.
You must identify the row with the most relevant retrieved document.
When you have identified the most relevant row, respond strictly in the format described
in the Output Format section.
"""


RESPONSE_TEMPLATE_CLASSIFICATION = """
Respond ONLY with a single JSON string, which is a list of boolean values the same length as
the DataFrame you received - in the JSON list, put a "1" at the element corresponding to the
most relevant row, and "0"s for all other elements.
Make sure to avoid backticks, using only single or double quotes throughout.
Ensure the square brackets defining the list start and end your response, do not deviate
from this. Take the format of the example below as a strict guide.
Example (in the case where there are 5 rows, and the second is the most relevant):
["0","1","0","0","0"]
"""

We can create a Hook for the AI agent which is a premade hook readily available with the package.

In [ ]:
import os
from classifai.indexers.hooks import RagHook


GOOGLE_PROJECT_ID = os.getenv("GOOGLE_PROJECT_ID")


# instantiate the agent hook agent first using the system prompts and api key for google cloud platform
classification_agent_hook = RagHook(
    project_id=GOOGLE_PROJECT_ID,
    vertexai=True,
    context_prompt=CONTEXT_PROMPT_CLASSIFICATION,
    response_template=RESPONSE_TEMPLATE_CLASSIFICATION,
)

apply the AI hook as we've seen with other hooks before when creating the vectorstore

In [ ]:
demo_vectorstore = VectorStore(
    file_name="./mock_coding_dataset.csv",
    data_type="csv",
    vectoriser=demo_vectoriser,
    output_dir="./demo_vdb",
    overwrite=True,
    hooks={"search_postprocess": classification_agent_hook},
)

call VectorStore.search() and see the AI agent in action.

In [ ]:
query_df = VectorStoreSearchInput({"id": [1], "query": ["organic cabbage farmer",]})
output = demo_vectorstore.search(query_df, n_results=5)


output

End Slide.